# Phase 4: Data Quality, Cleaning, and Preprocessing
## Customer Churn Prediction System

This notebook audits data quality anomalies, resolves missing whitespace in `Total Charges`, prevents data leakage by excluding post-churn and identifier columns, constructs a reproducible stratified train-test split, and defines a reusable Scikit-Learn `ColumnTransformer`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from churn_prediction.config import (
    ALL_FEATURE_COLS,
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET_COL,
    RANDOM_STATE,
    TEST_SIZE,
)
from churn_prediction.data_loader import load_raw_data
from churn_prediction.data_cleaning import clean_raw_dataset, clean_total_charges
from churn_prediction.data_validation import validate_raw_data, validate_cleaned_data, validate_train_test_split
from churn_prediction.preprocessing import build_preprocessor, prepare_and_split_data

### 1. Raw Data Quality Audit
We examine the raw dataset for missing values, whitespace, and duplicates.

In [2]:
df_raw = load_raw_data(use_cache=True)
validate_raw_data(df_raw)

# Investigate Total Charges whitespace
blank_tc = df_raw[df_raw["Total Charges"].astype(str).str.strip() == ""]
print(f"Total Charges blank whitespace count: {len(blank_tc)}")
display(blank_tc[["CustomerID", "Tenure Months", "Monthly Charges", "Total Charges"]])

Total Charges blank whitespace count: 11


,CustomerID,Tenure Months,Monthly Charges,Total Charges
2234,4472-LVYGI,0,52.55,
2438,3115-CZMZD,0,20.25,
2568,5709-LVOEQ,0,80.85,
2667,4367-NUYAO,0,25.75,
2856,1371-DWPAZ,0,56.05,
4331,7644-OMVMY,0,19.85,
4687,3213-VVOLG,0,25.35,
5104,2520-SGTTA,0,20.00,
5719,2923-ARZLG,0,19.70,
6772,4075-WKNIU,0,73.35,


### 2. Domain-Sound Cleaning
Notice that in all 11 records with blank `Total Charges`, `Tenure Months == 0`. These are brand-new customers who have not yet received their first bill.
We impute `0.0` for these accounts and convert `Total Charges` to `float64`.

In [3]:
df_clean = clean_raw_dataset(df_raw)
validate_cleaned_data(df_clean)
print("Cleaned Total Charges dtype:", df_clean["Total Charges"].dtype)
print("Null count in Total Charges post-cleaning:", df_clean["Total Charges"].isna().sum())

Cleaned Total Charges dtype: float64
Null count in Total Charges post-cleaning: 0


### 3. Leakage Prevention and Feature Scoping
We explicitly exclude:
- Post-churn feedback: `Churn Reason`
- Model-generated scores: `Churn Score`
- Derived metrics: `CLTV`
- Identifiers: `CustomerID`
- Zero-variance constants: `Count`, `Country`, `State`

In [4]:
print(f"Total Predictive Features: {len(ALL_FEATURE_COLS)}")
print("Numerical Features:", NUMERICAL_FEATURES)
print("Categorical Features:", CATEGORICAL_FEATURES)

Total Predictive Features: 19
Numerical Features: ['Tenure Months', 'Monthly Charges', 'Total Charges']
Categorical Features: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


### 4. Stratified Train-Test Splitting (Zero Data Leakage)
We split the data before fitting any transformations.

In [5]:
X_train, X_test, y_train, y_test = prepare_and_split_data(
    df_raw, test_size=TEST_SIZE, random_state=RANDOM_STATE, save_processed=True
)

print(f"X_train shape: {X_train.shape} (Churn rate: {y_train.mean():.4f})")
print(f"X_test shape:  {X_test.shape} (Churn rate: {y_test.mean():.4f})")

X_train shape: (5634, 19) (Churn rate: 0.2654)
X_test shape:  (1409, 19) (Churn rate: 0.2654)


### 5. Preprocessor Pipeline Verification
We build and inspect the `ColumnTransformer`.

In [6]:
preprocessor = build_preprocessor()
# Fit ONLY on training data
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"Transformed Training Matrix Shape: {X_train_trans.shape}")
print(f"Transformed Test Matrix Shape:     {X_test_trans.shape}")
print(f"Total One-Hot Encoded Features:    {len(feature_names)}")

Transformed Training Matrix Shape: (5634, 46)
Transformed Test Matrix Shape:     (1409, 46)
Total One-Hot Encoded Features:    46
